# FrictionLLM + RLCFrictionLM
### A neural architecture based on physical friction and RLC circuit mechanics

**Core idea:**
Current LLMs are pure resistors — every weight fires for every token.
This architecture maps real physics onto neurons:

| Physics | Component | Effect in the network |
|---|---|---|
| Static friction  | μ_s threshold  | Neuron stays **stuck at 0** unless signal exceeds threshold |
| Kinetic friction | μ_k drag       | Once fired, output = z − sign(z)·μ_k  (energy loss) |
| Inductance (L)   | Inertia        | Resists rapid changes in activation |
| Capacitance (C)  | Charge storage | Accumulates signal across **layers** before firing |
| Resonance        | ω₀ = 1/√(LC)  | Each neuron tunes to a natural frequency |

**What emerges:** sparsity (CPU-friendly inference), hierarchical feature tuning, and physically-principled dynamics.


## 1 · Install dependencies

In [ ]:
!pip install torch tiktoken numpy tqdm --quiet

## 2 · GPU setup

In [ ]:
import os, math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 3 · Write model source files

In [ ]:
import os
os.makedirs('friction_llm', exist_ok=True)

In [ ]:
%%writefile friction_llm/config.py
from dataclasses import dataclass, field


@dataclass
class FrictionConfig:
    # ── Vocabulary ────────────────────────────────────────────────────────────
    vocab_size: int = 50257      # GPT-2 tiktoken vocab

    # ── Architecture ─────────────────────────────────────────────────────────
    max_seq_len: int = 1024
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 6
    d_ff: int = 2048             # FFN hidden dim (4 × d_model recommended)
    dropout: float = 0.1
    bias: bool = True

    # ── Friction Gate ─────────────────────────────────────────────────────────
    # mu_s : static threshold  — force needed to START moving
    # mu_k : kinetic drag      — energy lost once moving  (always < mu_s)
    mu_s_init: float = 0.5
    mu_k_ratio_init: float = 0.6     # mu_k = mu_s * ratio,  ratio ∈ (0, 1)

    # Surrogate gradient sharpness (annealed from init → max during training)
    sharpness_init: float = 3.0
    sharpness_max: float = 50.0

    # ── Cross-Layer Momentum ─────────────────────────────────────────────────
    # Neurons "already in motion" in layer L have lower effective mu_s in L+1.
    use_momentum: bool = True
    momentum_alpha: float = 0.9      # exponential decay of momentum
    momentum_beta: float = 0.3       # how much momentum reduces mu_s (0–1)

    # ── RLC Circuit Neuron ───────────────────────────────────────────────────
    # Full RLC model: each neuron is an inductor (L) + resistor (R) + capacitor (C)
    # L : inductance  — inertia, resists changes in current
    # R : resistance  — our friction (dissipates energy)
    # C : capacitance — stores charge, creates restoring force
    # Natural frequency ω₀ = 1/√(LC), damping ratio ζ = R/(2√(L/C))
    use_rlc: bool = False               # switch from FGLU to RLC block
    rlc_dt: float = 0.1                 # Euler integration timestep
    rlc_L_init: float = 1.0             # initial inductance (per neuron)
    rlc_R_init: float = 2.0             # initial resistance — 2.0 = critical damping when L=C=1
    rlc_C_init: float = 1.0             # initial capacitance (per neuron)
    rlc_clamp: float = 10.0             # hard clamp on (q,i) to prevent blowup

    # ── Friction Attention (Phase 2 — off by default) ─────────────────────────
    use_friction_attention: bool = False
    attn_mu_s_init: float = 0.1
    attn_mu_k_ratio_init: float = 0.5

    # ── Training ─────────────────────────────────────────────────────────────
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    sparsity_reg: float = 0.0        # L1 penalty on gate activations

    # ── Curriculum: anneal sharpness smoothly so hard threshold emerges ───────
    curriculum_warmup_steps: int = 1000
    curriculum_anneal_steps: int = 10000

    # ── Hardware ─────────────────────────────────────────────────────────────
    use_amp: bool = True             # mixed precision (FP16 on CUDA)
    compile_model: bool = False      # torch.compile (PyTorch 2.x)

    # ── Checkpointing / Logging ───────────────────────────────────────────────
    checkpoint_dir: str = "checkpoints"
    save_every: int = 1000
    log_every: int = 100
    eval_every: int = 500
    eval_iters: int = 50

    # ── Quick-start presets ───────────────────────────────────────────────────
    @classmethod
    def tiny(cls) -> "FrictionConfig":
        """~1 M params — smoke test only, verifies code runs in seconds."""
        return cls(d_model=128, n_heads=4, n_layers=2, d_ff=512,
                   log_every=1, eval_every=9999, save_every=9999)

    @classmethod
    def small(cls) -> "FrictionConfig":
        """~30 M params — fast experiments on a single GPU or strong CPU."""
        return cls(d_model=384, n_heads=6, n_layers=6, d_ff=1536)

    @classmethod
    def medium(cls) -> "FrictionConfig":
        """~117 M params — GPT-2 scale, fits on RTX A6000 (48 GB)."""
        return cls(d_model=768, n_heads=12, n_layers=12, d_ff=3072)

    @classmethod
    def large(cls) -> "FrictionConfig":
        """~350 M params — training-scale experiment."""
        return cls(d_model=1024, n_heads=16, n_layers=24, d_ff=4096)


In [ ]:
%%writefile friction_llm/friction_gate.py
"""
Core friction mechanism: FrictionGate and FGLUBlock.

Physics model
─────────────
  |Z_i| ≤ μ_s  →  Y_i = 0            (stuck — static friction holds)
  |Z_i| >  μ_s  →  Y_i = Z_i − sign(Z_i)·μ_k  (moving — kinetic drag)

The "slip jolt" (μ_s > μ_k) creates a discontinuity — the neuron jumps
the moment it breaks free, then costs only μ_k to stay moving.

Training: smooth sigmoid surrogate (fully differentiable, sharpness annealed)
Inference: hard threshold (true physics, maximum sparsity → CPU wins big)
"""

import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class FrictionGate(nn.Module):
    """
    Learnable per-neuron static + kinetic friction gate.

    Parameters
    ──────────
    size             : number of neurons (d_ff)
    mu_s_init        : initial static threshold
    mu_k_ratio_init  : initial ratio mu_k / mu_s  ∈ (0, 1)  → ensures μ_k < μ_s
    sharpness        : surrogate gradient steepness (set externally by curriculum)
    momentum_alpha   : EMA decay for cross-layer momentum
    momentum_beta    : how much momentum lowers effective μ_s  (0 = none, 1 = full)
    use_momentum     : enable cross-layer momentum
    """

    def __init__(
        self,
        size: int,
        mu_s_init: float = 0.5,
        mu_k_ratio_init: float = 0.6,
        sharpness: float = 3.0,
        momentum_alpha: float = 0.9,
        momentum_beta: float = 0.3,
        use_momentum: bool = True,
    ) -> None:
        super().__init__()
        self.size = size
        self.sharpness = sharpness          # mutable — updated by SharpnessCurriculum
        self.use_momentum = use_momentum
        self.momentum_alpha = momentum_alpha
        self.momentum_beta = momentum_beta

        # ── μ_s = softplus(raw_μ_s)  →  always positive ──────────────────────
        raw_s = math.log(math.expm1(mu_s_init))  # inv-softplus
        self.raw_mu_s = nn.Parameter(torch.full((size,), raw_s))

        # ── μ_k = μ_s · sigmoid(raw_ratio)  →  μ_k ∈ (0, μ_s) always ────────
        raw_r = math.log(mu_k_ratio_init / (1.0 - mu_k_ratio_init))  # logit
        self.raw_ratio = nn.Parameter(torch.full((size,), raw_r))

    # ── Derived friction coefficients ────────────────────────────────────────

    @property
    def mu_s(self) -> torch.Tensor:
        return F.softplus(self.raw_mu_s)

    @property
    def mu_k(self) -> torch.Tensor:
        return self.mu_s * torch.sigmoid(self.raw_ratio)

    # ── Forward ──────────────────────────────────────────────────────────────

    def forward(
        self,
        z: torch.Tensor,
        momentum: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args
        ────
        z        : [..., size]   pre-activation signal ("applied force")
        momentum : [..., size]   cross-layer momentum from previous block, or None

        Returns
        ───────
        output       : [..., size]   friction-gated signal
        new_momentum : [..., size]   updated momentum (detached, no grad)
        """
        mu_s = self.mu_s   # [size]
        mu_k = self.mu_k   # [size]

        # Neurons "already in motion" have a lower effective threshold
        if self.use_momentum and momentum is not None:
            reduction = (self.momentum_beta * momentum).clamp(max=0.95)
            mu_s = (mu_s * (1.0 - reduction)).clamp(min=1e-6)

        output = self._smooth_forward(z, mu_s, mu_k) if self.training \
            else self._hard_forward(z, mu_s, mu_k)

        # Momentum update — no gradient flows through here
        with torch.no_grad():
            fired = (z.abs() > mu_s).float()
            if momentum is not None and self.use_momentum:
                new_momentum = self.momentum_alpha * momentum + (1.0 - self.momentum_alpha) * fired
            else:
                new_momentum = fired

        return output, new_momentum

    # ── Internal modes ───────────────────────────────────────────────────────

    def _smooth_forward(
        self, z: torch.Tensor, mu_s: torch.Tensor, mu_k: torch.Tensor
    ) -> torch.Tensor:
        """Differentiable surrogate: sigmoid ramp stands in for the step function."""
        gate = torch.sigmoid(self.sharpness * (z.abs() - mu_s))
        kinetic = z - z.sign() * mu_k
        return gate * kinetic

    def _hard_forward(
        self, z: torch.Tensor, mu_s: torch.Tensor, mu_k: torch.Tensor
    ) -> torch.Tensor:
        """True friction at inference — every zero is a real zero (skippable on CPU)."""
        mask = z.abs() > mu_s
        return torch.where(mask, z - z.sign() * mu_k, torch.zeros_like(z))

    # ── Diagnostics ──────────────────────────────────────────────────────────

    @torch.no_grad()
    def measure_sparsity(self, z: torch.Tensor) -> float:
        """Fraction of neurons zeroed. 0 = dense, 1 = fully sparse."""
        return (z.abs() <= self.mu_s).float().mean().item()

    def extra_repr(self) -> str:
        s = self.mu_s.mean().item()
        k = self.mu_k.mean().item()
        return f"size={self.size}, μ_s≈{s:.3f}, μ_k≈{k:.3f}, sharpness={self.sharpness}"


# ─────────────────────────────────────────────────────────────────────────────

class FGLUBlock(nn.Module):
    """
    Friction-Gated Linear Unit — drop-in replacement for the transformer FFN.

    Architecture (GLU-style)
    ────────────────────────
        gate_signal = W_gate(x)             [d_model → d_ff]  "applied force"
        value       = W_up(x)              [d_model → d_ff]  content
        gated       = FrictionGate(gate_signal)   sparse activation
        output      = W_out(gated ⊙ value)  [d_ff → d_model]

    Why GLU-style?
    ──────────────
    The gate controls *which* information passes; the value carries *what* it says.
    Sparsity in `gated` propagates into W_out — on CPU at inference, those rows
    of W_out can be skipped entirely (see inference.py sparse path).

    Weight count: 3 × d_model × d_ff  (vs 2 × for plain FFN).
    Set d_ff ≈ 2.67 × d_model to match parameter count with standard FFN.
    """

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        dropout: float = 0.1,
        mu_s_init: float = 0.5,
        mu_k_ratio_init: float = 0.6,
        sharpness: float = 3.0,
        momentum_alpha: float = 0.9,
        momentum_beta: float = 0.3,
        use_momentum: bool = True,
        bias: bool = True,
    ) -> None:
        super().__init__()
        self.W_gate = nn.Linear(d_model, d_ff, bias=bias)
        self.W_up   = nn.Linear(d_model, d_ff, bias=bias)
        self.W_out  = nn.Linear(d_ff,   d_model, bias=bias)
        self.drop   = nn.Dropout(dropout)

        self.friction = FrictionGate(
            d_ff,
            mu_s_init=mu_s_init,
            mu_k_ratio_init=mu_k_ratio_init,
            sharpness=sharpness,
            momentum_alpha=momentum_alpha,
            momentum_beta=momentum_beta,
            use_momentum=use_momentum,
        )

    def forward(
        self,
        x: torch.Tensor,
        momentum: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args
        ────
        x        : [B, T, d_model]
        momentum : [B, T, d_ff] or None

        Returns
        ───────
        output       : [B, T, d_model]
        new_momentum : [B, T, d_ff]
        """
        gate_signal = self.W_gate(x)                          # applied force
        value       = self.W_up(x)                            # content
        gated, new_momentum = self.friction(gate_signal, momentum)
        out = self.drop(self.W_out(gated * value))
        return out, new_momentum

    @torch.no_grad()
    def sparsity(self, x: torch.Tensor) -> float:
        """Gate sparsity for diagnostics (call in eval mode)."""
        return self.friction.measure_sparsity(self.W_gate(x))


In [ ]:
%%writefile friction_llm/attention.py
"""
Causal self-attention.

Phase 1: standard multi-head attention (flash attention if torch >= 2.0).
Phase 2: friction-sparse attention (toggled via config.use_friction_attention).
"""

from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import FrictionConfig
from .friction_gate import FrictionGate


class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention.

    Uses torch.nn.functional.scaled_dot_product_attention (flash attention)
    when available (PyTorch >= 2.0), otherwise falls back to manual O(T²).

    Phase 2 friction-sparse attention
    ──────────────────────────────────
    When config.use_friction_attention=True, a FrictionGate is applied to the
    raw attention logits before softmax. Logits below μ_s_attn are hard-zeroed;
    the surviving logits lose μ_k_attn of energy. This creates sparse attention
    patterns where low-relevance tokens are completely ignored.
    """

    def __init__(self, config: FrictionConfig) -> None:
        super().__init__()
        assert config.d_model % config.n_heads == 0, \
            "d_model must be divisible by n_heads"

        self.n_heads = config.n_heads
        self.d_head  = config.d_model // config.n_heads
        self.d_model = config.d_model

        self.qkv    = nn.Linear(config.d_model, 3 * config.d_model, bias=config.bias)
        self.proj   = nn.Linear(config.d_model, config.d_model,     bias=config.bias)
        self.drop_a = nn.Dropout(config.dropout)
        self.drop_r = nn.Dropout(config.dropout)

        # Phase 2: optional friction-sparse attention gate
        self.use_friction_attn = config.use_friction_attention
        if self.use_friction_attn:
            # One gate per head, gate operates on attention logits
            self.attn_friction = FrictionGate(
                size=config.n_heads,          # one threshold per head
                mu_s_init=config.attn_mu_s_init,
                mu_k_ratio_init=config.attn_mu_k_ratio_init,
                sharpness=config.sharpness_init,
                use_momentum=False,           # no cross-layer state in attention
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args
        ────
        x : [B, T, d_model]

        Returns
        ───────
        [B, T, d_model]
        """
        B, T, C = x.shape

        # Project to Q, K, V
        q, k, v = self.qkv(x).split(C, dim=-1)
        # Reshape to [B, heads, T, d_head]
        def reshape(t: torch.Tensor) -> torch.Tensor:
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = reshape(q), reshape(k), reshape(v)

        if self.use_friction_attn:
            out = self._friction_attention(q, k, v, B, T)
        elif hasattr(F, "scaled_dot_product_attention"):
            # Flash attention — O(T) memory, fastest path
            drop_p = self.drop_a.p if self.training else 0.0
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=drop_p, is_causal=True)
        else:
            out = self._manual_attention(q, k, v, T)

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop_r(self.proj(out))

    def _manual_attention(
        self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, T: int
    ) -> torch.Tensor:
        scale = self.d_head ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        causal_mask = torch.triu(torch.ones(T, T, device=q.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.drop_a(att)
        return att @ v

    def _friction_attention(
        self,
        q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
        B: int, T: int,
    ) -> torch.Tensor:
        """
        Phase 2: friction-gated sparse attention.

        Logits that don't have enough "force" to overcome μ_s_attn are zeroed —
        the model physically refuses to attend to weak relationships.
        """
        scale = self.d_head ** -0.5
        logits = (q @ k.transpose(-2, -1)) * scale    # [B, heads, T, T]

        # Apply causal mask
        causal_mask = torch.triu(
            torch.ones(T, T, device=q.device, dtype=torch.bool), diagonal=1
        )
        logits = logits.masked_fill(causal_mask, float("-inf"))

        # Friction gate on logits (per head, averaged across positions)
        # We reshape so friction operates per-head across the T×T logit space
        B_, H, T1, T2 = logits.shape
        logits_flat = logits.permute(0, 2, 3, 1).reshape(-1, H)   # [B·T·T, H]
        gated_flat, _ = self.attn_friction(logits_flat)            # sparse logits
        logits = gated_flat.reshape(B_, T1, T2, H).permute(0, 3, 1, 2)

        # Replace fully-zeroed rows with -inf so softmax doesn't give uniform
        all_zero_rows = (logits == 0).all(dim=-1, keepdim=True)
        logits = logits.masked_fill(all_zero_rows, float("-inf"))

        att = F.softmax(logits, dim=-1)
        att = torch.nan_to_num(att, nan=0.0)   # heads that attended nowhere → 0
        att = self.drop_a(att)
        return att @ v


In [ ]:
%%writefile friction_llm/block.py
"""
FrictionTransformerBlock — one layer of the friction LM.

Wiring
──────
    x  →  LayerNorm  →  CausalSelfAttention  →  residual
       →  LayerNorm  →  FGLUBlock(momentum_in)  →  residual
    momentum_in  (from previous block)  →  momentum_out  (to next block)
"""

from typing import Optional, Tuple

import torch
import torch.nn as nn

from .config import FrictionConfig
from .attention import CausalSelfAttention
from .friction_gate import FGLUBlock


class FrictionTransformerBlock(nn.Module):
    def __init__(self, config: FrictionConfig) -> None:
        super().__init__()
        self.ln1  = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2  = nn.LayerNorm(config.d_model)
        self.fglu = FGLUBlock(
            d_model=config.d_model,
            d_ff=config.d_ff,
            dropout=config.dropout,
            mu_s_init=config.mu_s_init,
            mu_k_ratio_init=config.mu_k_ratio_init,
            sharpness=config.sharpness_init,
            momentum_alpha=config.momentum_alpha,
            momentum_beta=config.momentum_beta,
            use_momentum=config.use_momentum,
            bias=config.bias,
        )

    def forward(
        self,
        x: torch.Tensor,
        momentum: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args
        ────
        x        : [B, T, d_model]
        momentum : [B, T, d_ff] or None   — from the previous block

        Returns
        ───────
        x            : [B, T, d_model]   updated hidden state
        new_momentum : [B, T, d_ff]      momentum for the next block
        """
        x = x + self.attn(self.ln1(x))
        fglu_out, new_momentum = self.fglu(self.ln2(x), momentum)
        x = x + fglu_out
        return x, new_momentum


In [ ]:
%%writefile friction_llm/model.py
"""
FrictionLM — full autoregressive language model.

Stack: token emb + positional emb → N × FrictionTransformerBlock → LayerNorm → LM head.
Weight tying: lm_head shares weights with token embedding (standard practice).
Momentum tensor is threaded through all blocks during a single forward pass.
"""

from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import FrictionConfig
from .block import FrictionTransformerBlock
from .friction_gate import FrictionGate


class FrictionLM(nn.Module):
    def __init__(self, config: FrictionConfig) -> None:
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop    = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(
            [FrictionTransformerBlock(config) for _ in range(config.n_layers)]
        )

        self.ln_f    = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying — halves embedding parameters, stabilises training
        self.lm_head.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self) -> None:
        """GPT-2-style initialisation: small normal, residual paths scaled by depth."""
        for name, p in self.named_parameters():
            if "weight" in name and p.dim() >= 2:
                nn.init.normal_(p, mean=0.0, std=0.02)
            elif "bias" in name:
                nn.init.zeros_(p)
        # Scale residual projections by 1/√(2·n_layers) — GPT-2 paper
        scale = (2 * self.config.n_layers) ** -0.5
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 * scale)
            nn.init.normal_(block.fglu.W_out.weight, std=0.02 * scale)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args
        ────
        idx     : [B, T]  token indices
        targets : [B, T]  next-token targets (None at inference)

        Returns
        ───────
        logits : [B, T, vocab_size]
        loss   : scalar cross-entropy loss, or None
        """
        B, T = idx.shape
        assert T <= self.config.max_seq_len, \
            f"Sequence length {T} exceeds max_seq_len {self.config.max_seq_len}"

        device = idx.device
        pos = torch.arange(T, device=device, dtype=torch.long)

        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        # Thread momentum through all blocks (None → first block initialises it)
        momentum: Optional[torch.Tensor] = None
        for block in self.blocks:
            x, momentum = block(x, momentum)

        x = self.ln_f(x)
        logits = self.lm_head(x)   # [B, T, vocab_size]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

    # ── Inference helpers ────────────────────────────────────────────────────

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
    ) -> torch.Tensor:
        """
        Autoregressive generation.

        Args
        ────
        idx            : [B, T]  seed token indices
        max_new_tokens : number of tokens to generate
        temperature    : sampling temperature (1.0 = unscaled)
        top_k          : if set, restrict sampling to top-k logits

        Returns
        ───────
        [B, T + max_new_tokens]
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

    # ── Diagnostics ──────────────────────────────────────────────────────────

    @torch.no_grad()
    def measure_sparsity(self, idx: torch.Tensor) -> dict:
        """
        Run a forward pass and return per-layer gate sparsity.
        Call in eval mode for hard-threshold (true) sparsity numbers.
        """
        self.eval()
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)

        stats = {}
        momentum = None
        for i, block in enumerate(self.blocks):
            gate_signal = block.fglu.W_gate(block.ln2(x))
            sparsity = block.fglu.friction.measure_sparsity(gate_signal)
            mu_s_mean = block.fglu.friction.mu_s.mean().item()
            mu_k_mean = block.fglu.friction.mu_k.mean().item()
            stats[f"layer_{i}"] = {
                "sparsity": sparsity,
                "mu_s": mu_s_mean,
                "mu_k": mu_k_mean,
            }
            x, momentum = block(x, momentum)

        overall = sum(v["sparsity"] for v in stats.values()) / len(stats)
        stats["overall"] = overall
        return stats

    def param_count(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def extra_repr(self) -> str:
        return (
            f"params={self.param_count()/1e6:.1f}M, "
            f"layers={self.config.n_layers}, "
            f"d_model={self.config.d_model}"
        )


In [ ]:
%%writefile friction_llm/curriculum.py
"""
SharpnessCurriculum — anneals the surrogate gradient sharpness over training.

Why this matters
────────────────
At low sharpness the friction gate is smooth (≈ soft-shrinkage).
The network trains easily but doesn't produce real sparsity.
As sharpness increases, the sigmoid ramp steepens toward a step function —
the "slip jolt" physics emerge, neurons snap to fully on/off, and true
sparsity grows.  By the end of curriculum_anneal_steps, inference can run
the hard-threshold path and get maximum sparsity for free.

Schedule: flat warmup → cosine anneal from sharpness_init to sharpness_max.
"""

import math

import torch.nn as nn

from .friction_gate import FrictionGate


class SharpnessCurriculum:
    """
    Controls how sharply the surrogate gradient approximates the step function.

    Usage
    ─────
        curriculum = SharpnessCurriculum(model, config)
        for step, batch in enumerate(dataloader):
            ...
            loss.backward()
            optimizer.step()
            sharpness = curriculum.step()   # call once per optimizer step
    """

    def __init__(self, model: nn.Module, config) -> None:
        self.model          = model
        self.sharpness_init = config.sharpness_init
        self.sharpness_max  = config.sharpness_max
        self.warmup_steps   = config.curriculum_warmup_steps
        self.anneal_steps   = config.curriculum_anneal_steps
        self._step          = 0

    def step(self) -> float:
        """Advance one optimizer step. Returns current sharpness."""
        self._step += 1
        sharpness = self._compute_sharpness(self._step)
        self._apply(sharpness)
        return sharpness

    def _compute_sharpness(self, step: int) -> float:
        if step <= self.warmup_steps:
            return self.sharpness_init
        progress = min((step - self.warmup_steps) / max(self.anneal_steps, 1), 1.0)
        # Cosine anneal: slow start, fast middle, slow end
        cosine = 0.5 * (1.0 - math.cos(math.pi * progress))
        return self.sharpness_init + (self.sharpness_max - self.sharpness_init) * cosine

    def _apply(self, sharpness: float) -> None:
        for module in self.model.modules():
            if isinstance(module, FrictionGate):
                module.sharpness = sharpness

    def state_dict(self) -> dict:
        return {"step": self._step}

    def load_state_dict(self, state: dict) -> None:
        self._step = state["step"]
        self._apply(self._compute_sharpness(self._step))


In [ ]:
%%writefile friction_llm/rlc_neuron.py
"""
RLC Circuit Neuron — full electromechanical physics mapped to neural computation.

Physics equations (series RLC / damped oscillator)
────────────────────────────────────────────────────
    L·q̈  +  R·q̇  +  q/C  =  V(t)

    L  inductance  ↔  mass       — inertia, resists change in current
    R  resistance  ↔  damping    — dissipates energy (our friction)
    C  capacitance ↔  1/spring   — stores charge, creates restoring force
    V(t)           ↔  applied force  — input signal from layer below
    q              ↔  position   — charge (neuron's accumulated state)
    i = dq/dt      ↔  velocity   — current (rate of change)

Rewritten as first-order system (state: q, i):
    dq/dt  =  i
    di/dt  =  ( V  −  R·i  −  q/C )  /  L

Integration: symplectic (semi-implicit) Euler — stable for oscillatory systems:
    i[t+1]  =  i[t]  +  di · dt        (update current first)
    q[t+1]  =  q[t]  +  i[t+1] · dt   (update charge with NEW current)

Three dynamical regimes — emerge naturally from learned L, R, C:
    ζ > 1  overdamped   : slow stable crawl, no oscillation
    ζ = 1  critical     : fastest response without overshoot  (init target)
    ζ < 1  underdamped  : rings at ω₀ = 1/√(LC), amplifies resonant inputs

Stacking across layers (depth = time)
──────────────────────────────────────
The circuit state (q, i) from layer L is passed as the initial state to layer
L+1.  Charge accumulated deep in the circuit seeds the next stage — like a
ladder network of RLC filters, each tuned to a different frequency band.
"""

import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .friction_gate import FrictionGate


# ─────────────────────────────────────────────────────────────────────────────

class RLCNeuron(nn.Module):
    """
    Per-neuron RLC circuit with learnable L, R, C.

    All three parameters are stored as log values → always positive.
    Initialized to critical damping (ζ = 1) by default, meaning the circuit
    reaches equilibrium as fast as possible without oscillating.  The network
    is free to learn underdamped (resonant) or overdamped neurons.

    State: (q, i) — charge and current — shape [..., size].
    Passed from one transformer block to the next (depth stacking).
    """

    def __init__(
        self,
        size: int,
        L_init: float = 1.0,
        R_init: float = 2.0,   # 2·√(L/C) = 2.0 → ζ = 1 when L=C=1
        C_init: float = 1.0,
        dt: float = 0.1,
        clamp: float = 10.0,
    ) -> None:
        super().__init__()
        self.size  = size
        self.dt    = dt
        self.clamp = clamp

        # log parameterisation keeps L, R, C strictly positive
        self.log_L = nn.Parameter(torch.full((size,), math.log(L_init)))
        self.log_R = nn.Parameter(torch.full((size,), math.log(R_init)))
        self.log_C = nn.Parameter(torch.full((size,), math.log(C_init)))

    # ── Circuit parameters ────────────────────────────────────────────────────

    @property
    def L(self) -> torch.Tensor:
        return self.log_L.exp()

    @property
    def R(self) -> torch.Tensor:
        return self.log_R.exp()

    @property
    def C(self) -> torch.Tensor:
        return self.log_C.exp()

    @property
    def omega_0(self) -> torch.Tensor:
        """Natural resonant frequency  ω₀ = 1 / √(LC)"""
        return 1.0 / (self.L * self.C).sqrt()

    @property
    def damping_ratio(self) -> torch.Tensor:
        """
        ζ = R / (2·√(L/C))
          < 1 : underdamped  (resonant — rings at ω₀)
          = 1 : critically damped (fastest stable response)
          > 1 : overdamped (slow, no oscillation)
        """
        return self.R / (2.0 * (self.L / self.C).sqrt())

    # ── Forward ───────────────────────────────────────────────────────────────

    def forward(
        self,
        V: torch.Tensor,
        state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Args
        ────
        V     : [..., size]  applied voltage (input force from W_gate projection)
        state : (q, i) or None — circuit state from previous layer

        Returns
        ───────
        q_new     : [..., size]  new charge (position)
        new_state : (q_new, i_new)
        """
        if state is None:
            q = torch.zeros_like(V)
            i = torch.zeros_like(V)
        else:
            q, i = state

        # di/dt = ( V − R·i − q/C ) / L
        di = (V - self.R * i - q / self.C) / self.L

        # Symplectic Euler: update i first, then q with new i
        i_new = i + di * self.dt
        q_new = q + i_new * self.dt

        # Clamp to prevent numerical blowup in early training
        i_new = i_new.clamp(-self.clamp, self.clamp)
        q_new = q_new.clamp(-self.clamp, self.clamp)

        return q_new, (q_new, i_new)

    # ── Diagnostics ──────────────────────────────────────────────────────────

    @torch.no_grad()
    def circuit_stats(self) -> dict:
        return {
            "omega_0_mean":  self.omega_0.mean().item(),
            "omega_0_std":   self.omega_0.std().item(),
            "zeta_mean":     self.damping_ratio.mean().item(),
            "zeta_std":      self.damping_ratio.std().item(),
            "underdamped_%": (self.damping_ratio < 1.0).float().mean().item() * 100,
            "overdamped_%":  (self.damping_ratio > 1.0).float().mean().item() * 100,
        }

    def extra_repr(self) -> str:
        ω = self.omega_0.mean().item()
        ζ = self.damping_ratio.mean().item()
        return f"size={self.size}, ω₀≈{ω:.3f}, ζ≈{ζ:.3f}, dt={self.dt}"


# ─────────────────────────────────────────────────────────────────────────────

class RLCFrictionBlock(nn.Module):
    """
    Full circuit neuron: RLC dynamics + friction gate + GLU projection.

    Signal path
    ───────────
        gate_signal = W_gate(x)          [d_model → d_ff]  "applied voltage"
        value       = W_up(x)            [d_model → d_ff]  content carrier

        q, (q,i) = RLC(gate_signal, prev_state)     circuit dynamics
        gated, momentum = FrictionGate(q, momentum) static/kinetic filter
                                                     on accumulated charge

        output = W_out( gated ⊙ value )  [d_ff → d_model]

    Why friction on charge (q), not on raw voltage (V)?
    ────────────────────────────────────────────────────
    A capacitor charges up over layers.  The friction gate fires only when
    accumulated charge exceeds μ_s — like a defibrillator capacitor that
    builds until it breaks through the threshold, then discharges with kinetic
    drag μ_k.  Single-shot voltage (V) hitting the gate would degrade to a
    plain threshold activation; charge (q) carries the circuit history.

    Stacking
    ────────
    Block returns (q_new, i_new) as new_state.  The next block receives this
    as its initial state — charge and current carry over, creating a ladder
    network of coupled RLC filters across depth.
    """

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        dropout: float = 0.1,
        mu_s_init: float = 0.5,
        mu_k_ratio_init: float = 0.6,
        sharpness: float = 3.0,
        momentum_alpha: float = 0.9,
        momentum_beta: float = 0.3,
        use_momentum: bool = True,
        bias: bool = True,
        L_init: float = 1.0,
        R_init: float = 2.0,
        C_init: float = 1.0,
        dt: float = 0.1,
        clamp: float = 10.0,
    ) -> None:
        super().__init__()
        self.W_gate = nn.Linear(d_model, d_ff, bias=bias)
        self.W_up   = nn.Linear(d_model, d_ff, bias=bias)
        self.W_out  = nn.Linear(d_ff,   d_model, bias=bias)
        self.drop   = nn.Dropout(dropout)

        self.rlc = RLCNeuron(
            size=d_ff,
            L_init=L_init, R_init=R_init, C_init=C_init,
            dt=dt, clamp=clamp,
        )
        self.friction = FrictionGate(
            size=d_ff,
            mu_s_init=mu_s_init,
            mu_k_ratio_init=mu_k_ratio_init,
            sharpness=sharpness,
            momentum_alpha=momentum_alpha,
            momentum_beta=momentum_beta,
            use_momentum=use_momentum,
        )

    def forward(
        self,
        x: torch.Tensor,
        rlc_state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        momentum: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor,
               Tuple[torch.Tensor, torch.Tensor],
               torch.Tensor]:
        """
        Args
        ────
        x         : [B, T, d_model]
        rlc_state : (q, i) each [B, T, d_ff], or None
        momentum  : [B, T, d_ff] or None

        Returns
        ───────
        output    : [B, T, d_model]
        new_state : (q_new, i_new)  — passed to next block
        new_mom   : [B, T, d_ff]   — passed to next block
        """
        V = self.W_gate(x)                                    # applied voltage
        q, new_state = self.rlc(V, rlc_state)                # circuit dynamics
        gated, new_mom = self.friction(q, momentum)           # friction on charge
        out = self.drop(self.W_out(gated * self.W_up(x)))     # GLU output
        return out, new_state, new_mom

    @torch.no_grad()
    def diagnostics(self, x: torch.Tensor) -> dict:
        """Per-block physics stats for monitoring during training."""
        V = self.W_gate(x)
        q, _ = self.rlc(V)
        sparsity = self.friction.measure_sparsity(q)
        stats = self.rlc.circuit_stats()
        stats["sparsity"] = sparsity
        stats["mu_s_mean"] = self.friction.mu_s.mean().item()
        stats["mu_k_mean"] = self.friction.mu_k.mean().item()
        return stats


In [ ]:
%%writefile friction_llm/rlc_block.py
"""
RLCTransformerBlock — one layer of the RLC circuit language model.

Threads three state tensors through the block stack:
    rlc_state : (q, i)   — charge and current  [B, T, d_ff]
    momentum  :          — friction momentum    [B, T, d_ff]

Both carry over from layer L to layer L+1, creating depth-as-time dynamics.
"""

from typing import Optional, Tuple

import torch
import torch.nn as nn

from .config import FrictionConfig
from .attention import CausalSelfAttention
from .rlc_neuron import RLCFrictionBlock


class RLCTransformerBlock(nn.Module):
    def __init__(self, config: FrictionConfig) -> None:
        super().__init__()
        self.ln1  = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2  = nn.LayerNorm(config.d_model)
        self.rlc_block = RLCFrictionBlock(
            d_model=config.d_model,
            d_ff=config.d_ff,
            dropout=config.dropout,
            mu_s_init=config.mu_s_init,
            mu_k_ratio_init=config.mu_k_ratio_init,
            sharpness=config.sharpness_init,
            momentum_alpha=config.momentum_alpha,
            momentum_beta=config.momentum_beta,
            use_momentum=config.use_momentum,
            bias=config.bias,
            L_init=config.rlc_L_init,
            R_init=config.rlc_R_init,
            C_init=config.rlc_C_init,
            dt=config.rlc_dt,
            clamp=config.rlc_clamp,
        )

    def forward(
        self,
        x: torch.Tensor,
        rlc_state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        momentum: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor,
               Tuple[torch.Tensor, torch.Tensor],
               torch.Tensor]:
        """
        Args
        ────
        x         : [B, T, d_model]
        rlc_state : (q, i) from previous block, or None
        momentum  : friction momentum from previous block, or None

        Returns
        ───────
        x         : [B, T, d_model]  updated hidden state
        new_state : (q_new, i_new)   circuit state for next block
        new_mom   : [B, T, d_ff]     friction momentum for next block
        """
        x = x + self.attn(self.ln1(x))
        rlc_out, new_state, new_mom = self.rlc_block(self.ln2(x), rlc_state, momentum)
        x = x + rlc_out
        return x, new_state, new_mom


In [ ]:
%%writefile friction_llm/rlc_model.py
"""
RLCFrictionLM — language model where every FFN is a full RLC circuit neuron.

What's new vs FrictionLM
────────────────────────
  FrictionLM   : FFN replaced by FGLUBlock  (friction gate — R only)
  RLCFrictionLM: FFN replaced by RLCFrictionBlock (L + R + C + friction gate)

Each neuron now carries:
  L  inductance  — inertia, resists rapid signal changes
  R  resistance  — our friction gate (static/kinetic threshold)
  C  capacitance — accumulates charge across layers

The circuit state (q=charge, i=current) propagates through all N blocks.
This turns depth into a physical time axis: charge builds through layers
like a wave moving through a ladder network of coupled RLC filters.

What emerges from training
──────────────────────────
  Different layers learn different natural frequencies ω₀ = 1/√(LC):
    Early layers → high ω₀ → high-pass → local / syntactic patterns
    Late layers  → low ω₀  → low-pass  → global / semantic patterns
  This is hierarchical feature extraction derived from physics — not just
  an empirical observation but a direct consequence of the circuit model.
"""

from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import FrictionConfig
from .rlc_block import RLCTransformerBlock
from .rlc_neuron import RLCNeuron
from .friction_gate import FrictionGate
from .curriculum import SharpnessCurriculum


class RLCFrictionLM(nn.Module):
    def __init__(self, config: FrictionConfig) -> None:
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop    = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(
            [RLCTransformerBlock(config) for _ in range(config.n_layers)]
        )

        self.ln_f    = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight   # weight tying

        self._init_weights()

    def _init_weights(self) -> None:
        for name, p in self.named_parameters():
            if "weight" in name and p.dim() >= 2:
                nn.init.normal_(p, mean=0.0, std=0.02)
            elif "bias" in name:
                nn.init.zeros_(p)
        scale = (2 * self.config.n_layers) ** -0.5
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,         std=0.02 * scale)
            nn.init.normal_(block.rlc_block.W_out.weight,   std=0.02 * scale)

    # ── Forward ──────────────────────────────────────────────────────────────

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args
        ────
        idx     : [B, T]  token indices
        targets : [B, T]  next-token targets, or None at inference

        Returns
        ───────
        logits : [B, T, vocab_size]
        loss   : cross-entropy scalar, or None
        """
        B, T = idx.shape
        assert T <= self.config.max_seq_len

        device = idx.device
        pos = torch.arange(T, device=device, dtype=torch.long)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        # Circuit state and friction momentum — both None at first block
        rlc_state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None
        momentum:  Optional[torch.Tensor] = None

        for block in self.blocks:
            x, rlc_state, momentum = block(x, rlc_state, momentum)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

    # ── Generation ───────────────────────────────────────────────────────────

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, -1:]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

    # ── Diagnostics ──────────────────────────────────────────────────────────

    @torch.no_grad()
    def circuit_report(self, idx: torch.Tensor) -> dict:
        """
        Run a forward pass and return per-layer circuit physics stats.

        Reports for each layer:
          ω₀   — natural frequency (should diverge after training)
          ζ    — damping ratio (1=critical, <1=resonant, >1=overdamped)
          sparsity — fraction of neurons zeroed by friction gate
          μ_s, μ_k — learned friction thresholds
        """
        self.eval()
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)

        report = {}
        rlc_state = None
        momentum  = None

        for i, block in enumerate(self.blocks):
            stats = block.rlc_block.diagnostics(block.ln2(x))
            report[f"layer_{i}"] = stats
            x, rlc_state, momentum = block(x, rlc_state, momentum)

        # Overall sparsity
        report["overall_sparsity"] = sum(
            v["sparsity"] for v in report.values() if isinstance(v, dict)
        ) / self.config.n_layers

        return report

    def param_count(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def extra_repr(self) -> str:
        return (
            f"params={self.param_count()/1e6:.1f}M  "
            f"layers={self.config.n_layers}  "
            f"d_model={self.config.d_model}  "
            f"RLC(L={self.config.rlc_L_init}, R={self.config.rlc_R_init}, C={self.config.rlc_C_init})"
        )

    def print_circuit_report(self, idx: torch.Tensor) -> None:
        """Pretty-print the per-layer circuit stats."""
        report = self.circuit_report(idx)

        print("\n── RLC Circuit Report ─────────────────────────────────────────────")
        print(f"{'Layer':<10} {'ω₀ mean':>10} {'ζ mean':>10} "
              f"{'Underdamp%':>12} {'Sparsity':>10} {'μ_s':>8} {'μ_k':>8}")
        print("─" * 72)

        for key, val in report.items():
            if not isinstance(val, dict):
                continue
            print(
                f"{key:<10} "
                f"{val['omega_0_mean']:>10.3f} "
                f"{val['zeta_mean']:>10.3f} "
                f"{val['underdamped_%']:>11.1f}% "
                f"{val['sparsity']:>9.1%} "
                f"{val['mu_s_mean']:>8.4f} "
                f"{val['mu_k_mean']:>8.4f}"
            )

        print("─" * 72)
        print(f"{'OVERALL':<10} {'':>10} {'':>10} {'':>12} "
              f"{report['overall_sparsity']:>9.1%}")
        print()


In [ ]:
%%writefile friction_llm/__init__.py
from .config import FrictionConfig
from .friction_gate import FrictionGate, FGLUBlock
from .attention import CausalSelfAttention
from .block import FrictionTransformerBlock
from .model import FrictionLM
from .curriculum import SharpnessCurriculum
from .rlc_neuron import RLCNeuron, RLCFrictionBlock
from .rlc_block import RLCTransformerBlock
from .rlc_model import RLCFrictionLM

__all__ = [
    "FrictionConfig",
    "FrictionGate",
    "FGLUBlock",
    "CausalSelfAttention",
    "FrictionTransformerBlock",
    "FrictionLM",
    "SharpnessCurriculum",
    "RLCNeuron",
    "RLCFrictionBlock",
    "RLCTransformerBlock",
    "RLCFrictionLM",
]


## 4 · Verify imports

In [ ]:
from friction_llm import (
    FrictionConfig, FrictionLM, RLCFrictionLM,
    SharpnessCurriculum, RLCNeuron
)
print('All imports OK')
cfg_test = FrictionConfig.tiny()
m_test = RLCFrictionLM(cfg_test)
print(f'Tiny RLC model: {m_test.param_count()/1e6:.1f}M params')

## 5 · Download & tokenise data

Using TinyShakespeare (~1 MB) — swap in any .txt corpus.

In [ ]:
import urllib.request, os

os.makedirs("data", exist_ok=True)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("data/input.txt"):
    urllib.request.urlretrieve(url, "data/input.txt")
    print("Downloaded TinyShakespeare")
else:
    print("Data already downloaded")
print(f"Size: {os.path.getsize('data/input.txt'):,} bytes")


In [ ]:
import tiktoken
import numpy as np

enc = tiktoken.get_encoding("gpt2")
with open("data/input.txt") as f:
    text = f.read()

tokens = np.array(enc.encode_ordinary(text), dtype=np.uint16)
split  = int(0.9 * len(tokens))
tokens[:split].tofile("data/train.bin")
tokens[split:].tofile("data/val.bin")
print(f"Train: {split:,} tokens   Val: {len(tokens)-split:,} tokens")


In [ ]:
class TokenDataset:
    def __init__(self, path, seq_len):
        self.data    = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len

    def get_batch(self, batch_size, device):
        ix = torch.randint(len(self.data) - self.seq_len - 1, (batch_size,))
        x  = torch.stack([torch.from_numpy(self.data[i:i+self.seq_len].astype(np.int64)) for i in ix])
        y  = torch.stack([torch.from_numpy(self.data[i+1:i+1+self.seq_len].astype(np.int64)) for i in ix])
        return x.to(device), y.to(device)

train_ds = TokenDataset("data/train.bin", seq_len=256)
val_ds   = TokenDataset("data/val.bin",   seq_len=256)
print(f"Batches available: {len(train_ds):,}")


## 6 · Training function

In [ ]:
from friction_llm import FrictionConfig, FrictionLM, RLCFrictionLM, SharpnessCurriculum

def train_model(model, train_ds, val_ds, max_steps=600, batch_size=16,
                lr=3e-4, log_every=50, label="model"):
    use_amp = device.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    # Separate physics params (no weight decay)
    physics, other = [], []
    for n, p in model.named_parameters():
        if any(k in n for k in ("raw_mu","raw_ratio","log_L","log_R","log_C")):
            physics.append(p)
        else:
            other.append(p)
    optimizer = torch.optim.AdamW(
        [{"params": other,   "weight_decay": 0.1},
         {"params": physics, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95)
    )

    curriculum = SharpnessCurriculum(model, model.config)
    history = {"step": [], "loss": [], "val_loss": [], "sparsity": []}
    best_val = float("inf")
    t0 = time.time()

    for step in range(max_steps + 1):
        # LR warmup
        warmup = 200
        cur_lr = lr * min(step / warmup, 1.0)
        for pg in optimizer.param_groups:
            pg["lr"] = cur_lr

        model.train()
        x, y = train_ds.get_batch(batch_size, device)
        with torch.amp.autocast("cuda", enabled=use_amp):
            _, loss = model(x, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        curriculum.step()

        if step % log_every == 0:
            # Validation
            model.eval()
            with torch.no_grad():
                xv, yv = val_ds.get_batch(batch_size, device)
                _, vloss = model(xv, yv)

            # Sparsity
            sample, _ = val_ds.get_batch(4, device)
            if hasattr(model, "circuit_report"):
                report  = model.circuit_report(sample)
                sparsity = report["overall_sparsity"]
            else:
                report  = model.measure_sparsity(sample)
                sparsity = report["overall"]
            model.train()

            dt = (time.time() - t0) / max(step, 1)
            history["step"].append(step)
            history["loss"].append(loss.item())
            history["val_loss"].append(vloss.item())
            history["sparsity"].append(sparsity)
            print(f"[{label}] step {step:4d} | loss {loss.item():.4f} "
                  f"| val {vloss.item():.4f} | sparsity {sparsity:.1%} "
                  f"| {dt*1000:.0f}ms/step")

    return history


## 7 · Train FrictionLM  (R only — static + kinetic friction)

The surrogate sharpness anneals from 3 → 50 over training.
Watch **sparsity** grow as the curriculum hardens the gate.

In [ ]:
cfg_f = FrictionConfig.small()
cfg_f.max_seq_len = 256
model_f = FrictionLM(cfg_f).to(device)
print(f"FrictionLM: {model_f.param_count()/1e6:.1f}M params")

history_f = train_model(model_f, train_ds, val_ds,
                        max_steps=600, batch_size=16, label="FrictionLM")


## 8 · Train RLCFrictionLM  (full L + R + C circuit)

Same architecture but the FFN is now a full RLC circuit neuron.
Each neuron has its own inductance L, resistance R, and capacitance C.
Charge accumulates across layers — the circuit fires when charge > μ_s.

In [ ]:
cfg_r = FrictionConfig.small()
cfg_r.max_seq_len = 256
cfg_r.use_rlc = True
model_r = RLCFrictionLM(cfg_r).to(device)
print(f"RLCFrictionLM: {model_r.param_count()/1e6:.1f}M params")

history_r = train_model(model_r, train_ds, val_ds,
                        max_steps=600, batch_size=16, label="RLCFrictionLM")


## 9 · Circuit physics report

Per-layer ω₀ (natural frequency) and ζ (damping ratio) after training.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
sample_text = "To be or not to be, that is the question"
sample_ids  = torch.tensor(enc.encode_ordinary(sample_text),
                           dtype=torch.long, device=device).unsqueeze(0)

model_r.eval()
model_r.print_circuit_report(sample_ids)


## 10 · Sparsity analysis

In [ ]:
# Per-layer sparsity breakdown for both models
model_f.eval(); model_r.eval()
sample, _ = val_ds.get_batch(8, device)

print("FrictionLM — per-layer sparsity:")
stats_f = model_f.measure_sparsity(sample)
for k, v in stats_f.items():
    if k == "overall":
        print(f"  OVERALL: {v:.1%}")
    else:
        print(f"  {k}: {v['sparsity']:.1%}  μ_s={v['mu_s']:.3f}  μ_k={v['mu_k']:.3f}")

print()
print("RLCFrictionLM — per-layer circuit stats:")
model_r.print_circuit_report(sample)


## 11 · Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
fig.suptitle("FrictionLLM vs RLCFrictionLM — Training Dynamics", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Loss curves ───────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(history_f["step"], history_f["val_loss"],  label="FrictionLM val",  color="#e05a5a", linewidth=2)
ax1.plot(history_r["step"], history_r["val_loss"],  label="RLCFrictionLM val", color="#5a9ce0", linewidth=2)
ax1.plot(history_f["step"], history_f["loss"],  linestyle="--", alpha=0.4, color="#e05a5a")
ax1.plot(history_r["step"], history_r["loss"],  linestyle="--", alpha=0.4, color="#5a9ce0")
ax1.set_xlabel("Step"); ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training & Validation Loss"); ax1.legend(); ax1.grid(alpha=0.3)

# ── Sparsity ──────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(history_f["step"], [s*100 for s in history_f["sparsity"]], color="#e05a5a", linewidth=2, label="FrictionLM")
ax2.plot(history_r["step"], [s*100 for s in history_r["sparsity"]], color="#5a9ce0", linewidth=2, label="RLCFrictionLM")
ax2.axhline(70, linestyle="--", color="gray", alpha=0.5, label="CPU-efficient threshold")
ax2.set_xlabel("Step"); ax2.set_ylabel("Sparsity (%)")
ax2.set_title("Gate Sparsity (higher = more neurons silent)"); ax2.legend(); ax2.grid(alpha=0.3)

# ── RLC natural frequencies per layer ────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
model_r.eval()
omega_per_layer, zeta_per_layer = [], []
for i, block in enumerate(model_r.blocks):
    rlc = block.rlc_block.rlc
    omega_per_layer.append(rlc.omega_0.detach().cpu().numpy())
    zeta_per_layer.append(rlc.damping_ratio.detach().cpu().numpy())

colors = plt.cm.viridis([i / len(omega_per_layer) for i in range(len(omega_per_layer))])
for i, (w, c) in enumerate(zip(omega_per_layer, colors)):
    ax3.hist(w, bins=30, alpha=0.6, color=c, label=f"Layer {i}")
ax3.set_xlabel("Natural Frequency ω₀"); ax3.set_ylabel("Count")
ax3.set_title("ω₀ Distribution per Layer\n(divergence = different frequency bands)"); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── Damping ratio per layer ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
for i, (z, c) in enumerate(zip(zeta_per_layer, colors)):
    ax4.hist(z, bins=30, alpha=0.6, color=c, label=f"Layer {i}")
ax4.axvline(1.0, color="red", linestyle="--", linewidth=2, label="Critical damping (ζ=1)")
ax4.set_xlabel("Damping Ratio ζ"); ax4.set_ylabel("Count")
ax4.set_title("Damping Ratio per Layer\n(<1=resonant, 1=critical, >1=overdamped)"); ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# ── μ_s per layer (friction thresholds) ──────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
mu_s_per_layer = []
for block in model_r.blocks:
    mu_s_per_layer.append(block.rlc_block.friction.mu_s.detach().cpu().numpy())
ax5.boxplot(mu_s_per_layer, labels=[f"L{i}" for i in range(len(mu_s_per_layer))])
ax5.set_xlabel("Layer"); ax5.set_ylabel("μ_s value")
ax5.set_title("Static Friction Thresholds μ_s\n(higher = harder to activate)"); ax5.grid(alpha=0.3)

plt.savefig("friction_rlc_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to friction_rlc_analysis.png")


## 12 · Text generation

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

def generate(model, prompt, max_tokens=150, temperature=0.8, top_k=40):
    model.eval()
    ids = enc.encode_ordinary(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(idx, max_new_tokens=max_tokens,
                         temperature=temperature, top_k=top_k)
    return enc.decode(out[0].tolist())

prompt = "HAMLET:\nTo be or not to be"

print("=" * 60)
print("FrictionLM output:")
print("=" * 60)
print(generate(model_f, prompt))

print()
print("=" * 60)
print("RLCFrictionLM output:")
print("=" * 60)
print(generate(model_r, prompt))


## What to try next

- **More steps**: `max_steps=20000` on an A6000 will get loss below 3.0
- **Friction attention**: set `config.use_friction_attention=True` to gate attention logits
- **Sparsity reg**: set `config.sparsity_reg=0.01` to push sparsity toward 80%+
- **Large corpus**: swap TinyShakespeare for WikiText-103 or OpenWebText
- **Ablation**: compare FrictionLM vs RLCFrictionLM at the same param count
